# 05 — The final model from the source code

The 33 features and the parameters chosen in notebook 04 now live in `src`. This notebook first checks that the source code gives the same result, and then asks what the final model can and cannot be trusted for. Nothing here changes the choice of features or parameters, so test scores are diagnostics, not selection.

In [1]:
%%time
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    precision_recall_curve,
    roc_auc_score,
)

from skeptical_second_reader.data import (
    ACCOUNTS_CSV,
    PATTERNS_TXT,
    account_entities,
    account_key,
    account_labels,
    load_transactions,
    pattern_members,
    read_accounts,
    read_patterns,
)
from skeptical_second_reader.explain import out_of_fold_shap, shap_values
from skeptical_second_reader.features import (
    ALL_FEATURES,
    FEATURES,
    WINDOW_END,
    exchange_rates,
    extended_features,
)
from skeptical_second_reader.model import fit_final, fit_xgboost
from skeptical_second_reader.paths import DATA_INTERIM
from skeptical_second_reader.split import (
    N_FOLDS,
    cached_folds,
    cv_predict,
    fold_roles,
    split_groups,
)

df = load_transactions()
labels = account_labels(df)
owners = account_entities(labels.index, read_accounts(ACCOUNTS_CSV))
patterns = read_patterns(PATTERNS_TXT)
members = pattern_members(patterns)
groups = split_groups(owners, members)
folds = cached_folds(labels, groups, DATA_INTERIM)

X = extended_features(df)
y = labels.loc[X.index]
fold_of = folds.loc[X.index]
print(f"model rows: {len(X):,}  positives: {y.sum():,}  features: {X.shape[1]}")

stored = pd.read_parquet(DATA_INTERIM / "experiment_features.parquet")
pd.testing.assert_frame_equal(X, stored[ALL_FEATURES], check_exact=True)
print("identical to the features of notebook 04: True")

model rows: 515,078  positives: 6,347  features: 33
identical to the features of notebook 04: True
CPU times: total: 1min 1s
Wall time: 1min


## 1. Does the source code reproduce notebook 04?

The per-round test AUC-PR of notebook 04 is written down here and compared. Thresholds use the same rule as in notebook 04: best F1 on the validation fold.

In [2]:
%%time


def best_f1_threshold(y_val, val_scores):
    precision, recall, thresholds = precision_recall_curve(y_val, val_scores)
    f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    return thresholds[f1.argmax()]


def per_round(scores, target, fold_labels):
    rows = []
    for k in range(N_FOLDS):
        test = fold_labels == k
        rows.append(
            {
                "auc_pr": average_precision_score(target[test], scores[test]),
                "auc_roc": roc_auc_score(target[test], scores[test]),
            }
        )
    return pd.DataFrame(rows)


def mean_sd(values):
    return f"{values.mean():.4f} +/- {values.std(ddof=1):.4f}"


models = []


def fit_and_keep(*args):
    score = fit_final(*args)
    models.append(score.model)
    return score


final = cv_predict(X, y, fold_of, fit_and_keep, best_f1_threshold)
baseline = cv_predict(X[FEATURES], y, fold_of, fit_xgboost)
final_rounds = per_round(final.scores, y, fold_of)
base_rounds = per_round(baseline.scores, y, fold_of)

NOTEBOOK_04 = [0.3812, 0.3451, 0.3552, 0.3601, 0.3777]
assert final_rounds["auc_pr"].round(4).tolist() == NOTEBOOK_04
print("per-round test AUC-PR equals notebook 04: True")
print("final AUC-PR:", mean_sd(final_rounds["auc_pr"]))
print("final AUC-ROC:", mean_sd(final_rounds["auc_roc"]))
print("baseline AUC-PR:", mean_sd(base_rounds["auc_pr"]))

per-round test AUC-PR equals notebook 04: True
final AUC-PR: 0.3639 +/- 0.0153
final AUC-ROC: 0.9109 +/- 0.0072
baseline AUC-PR: 0.2893 +/- 0.0218
CPU times: total: 25min 32s
Wall time: 1min 45s


## 2. What do the cross-account inputs contribute?

The exchange rates and the counterparty degrees are computed over all accounts, without labels. Here both are limited, per round, to transactions between training accounts. A counterparty unknown to that graph counts as degree 0. This is harsher than deployment on current customers, and it creates a small mismatch: a training account's own links are in the graph, a test account's are not.

In [3]:
%%time
sender = account_key(df["from_bank"], df["from_account"])
receiver = account_key(df["to_bank"], df["to_account"])
full_rates = exchange_rates(df)

limited_rows, rate_shifts = [], []
for k in range(N_FOLDS):
    roles = fold_roles(fold_of, k)
    train_accounts = roles.index[roles == "train"]
    seen = df[sender.isin(train_accounts) & receiver.isin(train_accounts)]
    rates = exchange_rates(seen)
    rate_shifts.append(((rates / full_rates).sub(1).abs()).max())
    variants = {
        "rates from training accounts": {"rates": rates},
        "degrees from training accounts": {"degree_transactions": seen},
        "both": {"rates": rates, "degree_transactions": seen},
    }
    train, val, test = roles == "train", roles == "val", roles == "test"
    for name, inputs in variants.items():
        table = extended_features(df, **inputs)
        score = fit_final(table[train], y[train], table[val], y[val])
        limited_rows.append(
            {
                "variant": name,
                "round": k,
                "auc_pr": average_precision_score(y[test], score(table[test])),
                "gain": average_precision_score(y[test], score(table[test]))
                - final_rounds.loc[k, "auc_pr"],
            }
        )
limited = pd.DataFrame(limited_rows)
print(
    limited.groupby("variant")[["auc_pr", "gain"]]
    .agg(["mean", "std"])
    .round(4)
    .to_string()
)
print("largest relative change of any rate, per round:", np.round(rate_shifts, 4))

                                auc_pr            gain        
                                  mean     std    mean     std
variant                                                       
both                            0.3008  0.0222 -0.0630  0.0091
degrees from training accounts  0.3009  0.0210 -0.0630  0.0081
rates from training accounts    0.3645  0.0159  0.0007  0.0009
largest relative change of any rate, per round: [0.0006 0.0001 0.     0.     0.001 ]
CPU times: total: 1h 9min 16s
Wall time: 16min 58s


Limiting the degrees has a mismatch built in, as noted above. Two further variants separate that mismatch from a real loss. In the first, the graph still has training accounts only, but an account's own link is taken out of its counterparty's degree, so training and test accounts are computed alike. In the second, the two degree features are removed altogether, which needs no graph.

In [4]:
%%time
in_window = df["timestamp"] < WINDOW_END
is_self = sender == receiver
links = pd.DataFrame(
    {"s": sender[in_window & ~is_self], "r": receiver[in_window & ~is_self]}
).drop_duplicates()


def degrees_without_own_link(train_accounts):
    in_graph = links["s"].isin(train_accounts) & links["r"].isin(train_accounts)
    graph = links[in_graph]
    own = in_graph.astype(int)
    to_receiver = links["r"].map(graph["r"].value_counts()).fillna(0) - own
    to_sender = links["s"].map(graph["s"].value_counts()).fillna(0) - own
    assert to_receiver.min() >= 0 and to_sender.min() >= 0
    return pd.DataFrame(
        {
            "mean_receiver_in_degree": to_receiver.groupby(links["s"]).mean(),
            "mean_sender_out_degree": to_sender.groupby(links["r"]).mean(),
        }
    ).reindex(X.index, fill_value=0.0)


DEGREES = ["mean_receiver_in_degree", "mean_sender_out_degree"]
kept_columns = [column for column in ALL_FEATURES if column not in DEGREES]
degree_rows, balance_rows = [], []
for k in range(N_FOLDS):
    roles = fold_roles(fold_of, k)
    train, val, test = roles == "train", roles == "val", roles == "test"
    degrees = degrees_without_own_link(roles.index[train])
    balance_rows.append(
        {
            "round": k,
            "train_mean": degrees.loc[train, DEGREES[0]].mean(),
            "test_mean": degrees.loc[test, DEGREES[0]].mean(),
        }
    )
    tables = {
        "training graph, own link removed": X.assign(**degrees),
        "without the two degree features": X[kept_columns],
    }
    for name, table in tables.items():
        score = fit_final(table[train], y[train], table[val], y[val])
        auc_pr = average_precision_score(y[test], score(table[test]))
        degree_rows.append(
            {
                "variant": name,
                "auc_pr": auc_pr,
                "gain": auc_pr - final_rounds.loc[k, "auc_pr"],
            }
        )
print(
    pd.DataFrame(degree_rows)
    .groupby("variant")[["auc_pr", "gain"]]
    .agg(["mean", "std"])
    .round(4)
    .to_string()
)
print()
print(pd.DataFrame(balance_rows).round(3).to_string(index=False))

                                  auc_pr            gain        
                                    mean     std    mean     std
variant                                                         
training graph, own link removed  0.3245  0.0210 -0.0393  0.0085
without the two degree features   0.3446  0.0192 -0.0192  0.0067

 round  train_mean  test_mean
     0       2.187      1.753
     1       2.560      2.272
     2       2.449      1.878
     3       2.359      1.949
     4       2.133      1.815
CPU times: total: 41min 51s
Wall time: 3min 16s


## 3. How much of the signal is the labelled transactions themselves?

The features can see the laundering transactions of the window. Here all labelled transactions are removed before the features are computed (an oracle removal, impossible in practice). What remains shows how far the model recognises an account by its other behaviour. Accounts with only labelled transactions lose their row, so both scores are computed on the same remaining accounts.

In [5]:
%%time
clean_X = extended_features(df[df["is_laundering"] == 0])
common = X.index.intersection(clean_X.index)
clean_X, clean_y, clean_folds = clean_X.loc[common], y.loc[common], fold_of.loc[common]
clean_run = cv_predict(clean_X, clean_y, clean_folds, fit_final)
clean_rounds = per_round(clean_run.scores, clean_y, clean_folds)
same_accounts = per_round(final.scores.loc[common], clean_y, clean_folds)
print(f"accounts kept: {len(common):,} of {len(X):,}")
print(f"positives kept: {clean_y.sum():,} of {y.sum():,}")
print(f"positive rate of the kept accounts: {clean_y.mean():.4f}")
print(
    "AUC-PR, features without labelled transactions:", mean_sd(clean_rounds["auc_pr"])
)
print("AUC-PR, final model, same accounts:", mean_sd(same_accounts["auc_pr"]))
print(
    "AUC-ROC, features without labelled transactions:", mean_sd(clean_rounds["auc_roc"])
)
print("AUC-ROC, final model, same accounts:", mean_sd(same_accounts["auc_roc"]))

accounts kept: 515,011 of 515,078
positives kept: 6,280 of 6,347
positive rate of the kept accounts: 0.0122
AUC-PR, features without labelled transactions: 0.2535 +/- 0.0263
AUC-PR, final model, same accounts: 0.3652 +/- 0.0153
AUC-ROC, features without labelled transactions: 0.8128 +/- 0.0033
AUC-ROC, final model, same accounts: 0.9125 +/- 0.0074
CPU times: total: 20min 35s
Wall time: 2min 12s


## 4. Thresholds

In notebook 04, round 0 flagged more accounts at a lower precision. Is the F1 curve flat around the chosen threshold, so that small validation differences move it far? Two other validation-only rules are compared: the lowest threshold with validation precision of at least 0.7, and the top 500 validation scores.

In [6]:
def f1_at(target, scores, threshold):
    flagged = scores >= threshold
    hits = target[flagged].sum()
    return 2 * hits / (flagged.sum() + target.sum())


def precision_rule(y_val, val_scores, wanted=0.7):
    precision, _, thresholds = precision_recall_curve(y_val, val_scores)
    return thresholds[np.argmax(precision[:-1] >= wanted)]


def budget_rule(y_val, val_scores, budget=500):
    return np.sort(val_scores)[-budget]


RULES = {
    "best F1": best_f1_threshold,
    "precision 0.7": precision_rule,
    "top 500": budget_rule,
}
flat_rows, rule_rows = [], []
for k, model in enumerate(models):
    roles = fold_roles(fold_of, k)
    val, test = roles == "val", roles == "test"
    val_scores = model.predict_proba(X[val])[:, 1]
    own = final.thresholds[k]
    others = np.median([t for i, t in enumerate(final.thresholds) if i != k])
    flat_rows.append(
        {
            "round": k,
            "threshold": own,
            "val_f1": f1_at(y[val], val_scores, own),
            "median_of_other_thresholds": others,
            "val_f1_there": f1_at(y[val], val_scores, others),
        }
    )
    for name, rule in RULES.items():
        threshold = rule(y[val], val_scores)
        flagged = final.scores[test] >= threshold
        rule_rows.append(
            {
                "rule": name,
                "round": k,
                "flagged": flagged.sum(),
                "precision": y[test][flagged].mean(),
                "recall": flagged[y[test] == 1].mean(),
            }
        )
print(pd.DataFrame(flat_rows).round(4).to_string(index=False))
rules = pd.DataFrame(rule_rows)
print()
print(
    rules.groupby("rule")[["flagged", "precision", "recall"]]
    .agg(["mean", "min", "max"])
    .round(3)
    .to_string()
)

 round  threshold  val_f1  median_of_other_thresholds  val_f1_there
     0     0.1454  0.3605                      0.2353        0.3300
     1     0.2218  0.3774                      0.2353        0.3731
     2     0.2661  0.3882                      0.2230        0.3817
     3     0.2464  0.3880                      0.2230        0.3819
     4     0.2242  0.4057                      0.2341        0.3993

              flagged           precision               recall              
                 mean  min  max      mean    min    max   mean    min    max
rule                                                                        
best F1         476.6  404  693     0.715  0.561  0.778  0.262  0.229  0.306
precision 0.7   467.8  447  484     0.706  0.678  0.719  0.260  0.239  0.272
top 500         500.2  459  565     0.675  0.631  0.739  0.265  0.250  0.286


## 5. Calibration

The LLM will read the score. If it is shown as a probability, does a score of 0.8 mean that about 80% of such accounts are positive? Out-of-fold scores, all rounds together.

In [7]:
bins = [0, 0.01, 0.05, 0.1, 0.2, 0.4, 0.6, 0.8, 0.9, 1.0]
binned = pd.cut(final.scores, bins, include_lowest=True)
calibration = pd.DataFrame(
    {
        "accounts": final.scores.groupby(binned, observed=False).size(),
        "mean_score": final.scores.groupby(binned, observed=False).mean(),
        "positive_rate": y.groupby(binned, observed=False).mean(),
    }
)
print(calibration.round(4).to_string())
print(f"\nBrier score: {brier_score_loss(y, final.scores):.5f}")
print(
    f"Brier score of always predicting the base rate: {y.mean() * (1 - y.mean()):.5f}"
)
print(f"mean score: {final.scores.mean():.4f}  positive rate: {y.mean():.4f}")

                accounts  mean_score  positive_rate
score                                              
(-0.001, 0.01]    396527      0.0021         0.0019
(0.01, 0.05]       95561      0.0213         0.0237
(0.05, 0.1]        15926      0.0689         0.0678
(0.1, 0.2]          4628      0.1317         0.1255
(0.2, 0.4]           833      0.2635         0.2821
(0.4, 0.6]           226      0.4978         0.5796
(0.6, 0.8]           225      0.7056         0.7689
(0.8, 0.9]           211      0.8524         0.9194
(0.9, 1.0]           941      0.9609         0.9883

Brier score: 0.00923
Brier score of always predicting the base rate: 0.01217
mean score: 0.0120  positive rate: 0.0123


## 6. Which positives are missed, and who are the false positives?

Recall at the validation threshold and the median rank among all accounts, by what is known about the positive. An account can appear in several pattern types. The false positives are compared with all negatives: are they close to laundering, through a counterparty or an owner?

In [8]:
ranks = final.scores.rank(pct=True)
flags = final.flags

in_window = df["timestamp"] < WINDOW_END
is_self = sender == receiver
bad = in_window & (df["is_laundering"] == 1)
bad_touches = pd.concat([sender[bad], receiver[bad & ~is_self]])
bad_count = bad_touches.value_counts().reindex(X.index, fill_value=0)

ach_bad = bad & (df["payment_format"] == "ACH")
ach_accounts = set(sender[ach_bad]) | set(receiver[ach_bad])


def describe(mask):
    rows = y.index[mask & (y == 1)]
    return {
        "positives": len(rows),
        "recall": flags[rows].mean(),
        "median_rank": ranks[rows].median(),
    }


groups_of_positives = {
    "no labelled tx in the window": bad_count == 0,
    "1 labelled tx": bad_count == 1,
    "2-5 labelled tx": bad_count.between(2, 5),
    "6+ labelled tx": bad_count >= 6,
    "has an ACH labelled tx": X.index.isin(ach_accounts),
    "labelled tx, none ACH": (bad_count > 0) & ~X.index.isin(ach_accounts),
    "not in the pattern file": ~X.index.isin(members["account"]),
}
kinds = patterns.drop_duplicates("attempt").set_index("attempt")["pattern"]
typed = members.assign(pattern=members["attempt"].map(kinds))
for kind, part in typed.groupby("pattern"):
    groups_of_positives[f"pattern: {kind}"] = X.index.isin(part["account"])
print(
    pd.DataFrame({k: describe(m) for k, m in groups_of_positives.items()})
    .T.round(3)
    .to_string()
)

without_tail_only = (bad_count > 0) | (y == 0)
print(
    "\nAUC-PR without positives lacking a labelled tx in the window:",
    mean_sd(
        per_round(
            final.scores[without_tail_only],
            y[without_tail_only],
            fold_of[without_tail_only],
        )["auc_pr"]
    ),
)

                              positives  recall  median_rank
no labelled tx in the window      485.0   0.002        0.637
1 labelled tx                    4844.0   0.183        0.953
2-5 labelled tx                   878.0   0.736        0.998
6+ labelled tx                    140.0   0.936        0.998
has an ACH labelled tx           5122.0   0.321        0.977
labelled tx, none ACH             740.0   0.023        0.866
not in the pattern file          3187.0   0.026        0.928
pattern: BIPARTITE                491.0   0.393        0.989
pattern: CYCLE                    270.0   0.707        0.998
pattern: FAN-IN                   336.0   0.571        0.997
pattern: FAN-OUT                  356.0   0.396        0.990
pattern: GATHER-SCATTER           683.0   0.425        0.982
pattern: RANDOM                   211.0   0.564        0.997
pattern: SCATTER-GATHER           368.0   0.715        0.998
pattern: STACK                    662.0   0.503        0.996



AUC-PR without positives lacking a labelled tx in the window: 0.3890 +/- 0.0145


In [9]:
positives = set(y.index[y == 1])
window_pairs = pd.DataFrame(
    {"a": sender[in_window & ~is_self], "b": receiver[in_window & ~is_self]}
).drop_duplicates()
near = set(window_pairs.loc[window_pairs["b"].isin(positives), "a"]) | set(
    window_pairs.loc[window_pairs["a"].isin(positives), "b"]
)
has_positive_neighbour = pd.Series(X.index.isin(near), index=X.index)

account_owner = owners.reindex(X.index)
positive_owners = set(account_owner[y == 1])
shares_owner = account_owner.isin(positive_owners)

negatives = y == 0
false_positive = negatives & (flags == 1)
top_negative = negatives & (ranks >= ranks[negatives].quantile(0.999))
comparison = pd.DataFrame(
    {
        name: {
            "accounts": mask.sum(),
            "positive_counterparty": has_positive_neighbour[mask].mean(),
            "owner_has_a_positive": shares_owner[mask].mean(),
            "n_currencies_2_or_more": (X.loc[mask, "n_currencies"] >= 2).mean(),
            "any_ach_out": (X.loc[mask, "ach_out_tx"] > 0).mean(),
        }
        for name, mask in {
            "all negatives": negatives,
            "false positives": false_positive,
            "top 0.1% of negatives": top_negative,
            "true positives": (y == 1) & (flags == 1),
            "missed positives": (y == 1) & (flags == 0),
        }.items()
    }
)
print(comparison.round(3).to_string())

                        all negatives  false positives  top 0.1% of negatives  true positives  missed positives
accounts                   508731.000          720.000                509.000        1663.000          4684.000
positive_counterparty           0.113            0.275                  0.267           1.000             0.912
owner_has_a_positive            0.238            0.217                  0.208           1.000             1.000
n_currencies_2_or_more          0.015            0.204                  0.271           0.711             0.020
any_ach_out                     0.322            0.781                  0.796           0.898             0.606


## 7. Are the explanations stable?

The LLM will read SHAP values, so they should not depend much on which round's model explains an account. All five models explain the same sample of accounts. The models share training data, so this measures stability under a changed training set, not independence.

In [10]:
%%time
explained = out_of_fold_shap(models, X, fold_of)
only_features = explained[ALL_FEATURES]
importance = only_features.abs().mean().sort_values(ascending=False)
print(importance.head(5).round(4).to_string())
assert abs(importance["ach_in_tx"] - 0.5857) < 1e-4
print("agrees with notebook 04: True\n")

top_scored = final.scores.nlargest(2000).index
others = final.scores.drop(top_scored).sample(2000, random_state=0).index
sample = top_scored.append(others)
per_model = [shap_values(m, X.loc[sample])[ALL_FEATURES] for m in models]

rank_corr = pd.DataFrame({k: v.abs().mean() for k, v in enumerate(per_model)}).corr(
    method="spearman"
)
off_diagonal = rank_corr.to_numpy()[~np.eye(N_FOLDS, dtype=bool)]
print(f"rank correlation of feature importance, lowest pair: {off_diagonal.min():.3f}")


def top_sets(values, n):
    order = np.argsort(-values.abs().to_numpy(), axis=1)[:, :n]
    return [set(row) for row in order]


for name, part in {"top-scored accounts": top_scored, "other accounts": others}.items():
    position = sample.get_indexer(part)
    sets = [top_sets(v.iloc[position], 3) for v in per_model]
    overlaps = [
        np.mean([len(p & q) / 3 for p, q in zip(sets[i], sets[j], strict=True)])
        for i in range(N_FOLDS)
        for j in range(i + 1, N_FOLDS)
    ]
    print(
        f"{name}: top-3 feature overlap between two models, "
        f"mean {np.mean(overlaps):.3f}, lowest pair {np.min(overlaps):.3f}"
    )

ach_in_tx                 0.5857
mean_sender_out_degree    0.3414
ach_receivers             0.3067
log_max_sent_usd          0.2904
in_tx                     0.2705
agrees with notebook 04: True



rank correlation of feature importance, lowest pair: 0.939
top-scored accounts: top-3 feature overlap between two models, mean 0.936, lowest pair 0.925
other accounts: top-3 feature overlap between two models, mean 0.766, lowest pair 0.699
CPU times: total: 26min 32s
Wall time: 1min 44s


## 8. Files for the next step, and candidate cases

Out-of-fold scores, flags and SHAP values go to `data/interim`, which is not in the repository, for the interface and the LLM step. Candidate cases are drawn at random from five kinds of model outcome. They are a proposal for the first slice, not the experiment's final sample. Only counts and score ranges are shown here.

In [11]:
threshold_of = fold_of.map(dict(enumerate(final.thresholds)))
results = pd.DataFrame(
    {
        "score": final.scores,
        "threshold": threshold_of,
        "flag": flags,
        "label": y,
        "fold": fold_of,
    }
)
results.to_parquet(DATA_INTERIM / "final_scores.parquet")
explained.to_parquet(DATA_INTERIM / "final_shap.parquet")


def logit(values):
    clipped = values.clip(1e-6, 1 - 1e-6)
    return np.log(clipped / (1 - clipped))


distance = (logit(results["score"]) - logit(results["threshold"])).abs()
pools = {
    "confident true positive": results[
        results["flag"].eq(1) & results["label"].eq(1)
    ].nlargest(500, "score"),
    "confident false positive": results[
        results["flag"].eq(1) & results["label"].eq(0)
    ].nlargest(100, "score"),
    "missed positive": results[
        results["flag"].eq(0) & results["label"].eq(1)
    ].nsmallest(500, "score"),
    "clear negative": results[results["flag"].eq(0) & results["label"].eq(0)].nsmallest(
        5000, "score"
    ),
    "near the threshold": results.loc[distance.nsmallest(200).index],
}
cases = pd.concat(
    [pool.sample(2, random_state=0).assign(kind=kind) for kind, pool in pools.items()]
)
cases.to_parquet(DATA_INTERIM / "candidate_cases.parquet")
print(
    cases.groupby("kind", sort=False)
    .agg(
        cases=("score", "size"),
        lowest_score=("score", "min"),
        highest_score=("score", "max"),
        positives=("label", "sum"),
        flagged=("flag", "sum"),
    )
    .round(4)
    .to_string()
)

                          cases  lowest_score  highest_score  positives  flagged
kind                                                                            
confident true positive       2        0.9799         0.9892          2        2
confident false positive      2        0.5787         0.8017          0        2
missed positive               2        0.0011         0.0030          2        0
clear negative                2        0.0000         0.0000          0        0
near the threshold            2        0.2674         0.2743          0        2
